<a href="https://colab.research.google.com/github/impriyanka60/AiLearningTutor/blob/main/Merged_Pipeline_(with_interpolation).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Merged GPS Field Boundary Pipeline
DBSCAN (auto-eps) decides the number of clusters. If DBSCAN gives more than 1 cluster, KMeans and DePDDP are also run with that same k. Angle-based start/end trail removal runs on **every** algorithm's clusters. The algorithm with the highest Calinski-Harabasz (CH) index (computed on the cleaned/no-tail data) is selected. Point injection (d=r) + concave hull (tol=3) + polylabel centroid are computed **only** for the winning algorithm's clusters.

In [ ]:
!pip install pyproj kneed python-polylabel cdBoundary HiPart

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.3/717.3 kB 37.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score

from pyproj import Transformer
from kneed import KneeLocator
from shapely.geometry import Polygon
from polylabel import polylabel
from cdBoundary.boundary import ConcaveHull
from HiPart.clustering import DePDDP

## Config

In [ ]:
DATA_FOLDER   = "/content/drive/MyDrive/moong_26/24thJune/csv"
OUTPUT_FOLDER = "/content/drive/MyDrive/MergedPipeline(0.75r)newdata/24thJune/tol(2.5)"

MAX_FILES = 80
ALPHA     = 1          # weight applied to (scaled) time axis
NEIGHBORS = 15         # kNN neighbors for DBSCAN eps auto-detection & min_samples
ANGLE_THRESHOLD = 80   # deg, for start/end trail removal
MIN_CLUSTER_POINTS = 10
MIN_HULL_CLUSTER_POINTS = 15  # clusters of the *winning* algorithm smaller than this are dropped before injection/hull
HULL_TOL_MULTIPLIER = 2.5 # tol = ch.estimate() * HULL_TOL_MULTIPLIER (adaptive per-cluster, not a fixed number)
INTERP_STEP_SECONDS = 5   # linear interpolation time step (sec), applied to cleaned cluster points before inject-point step

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

## Prepare data

In [ ]:
def prepare_data(df):
    lat = df["Lat"].to_numpy()
    lon = df["Lon"].to_numpy()

    if "Time" in df.columns:
        time = df["Time"].to_numpy()
        time = time - time.min()
    else:
        time = np.arange(len(df))

    easting, northing = transformer.transform(lon, lat)
    utm = np.column_stack((easting, northing))

    combined = np.column_stack((utm, time))
    scaled = MinMaxScaler().fit_transform(combined)
    scaled[:, 2] *= ALPHA

    return utm, scaled, time

## Cluster quality score (Silhouette + Calinski-Harabasz)

In [ ]:
def compute_scores(X, labels):
    unique_labels = set(labels)

    if unique_labels == {-1}:
        return None, None, "All noise"

    mask = labels != -1
    X_clean = X[mask]
    labels_clean = labels[mask]

    if len(np.unique(labels_clean)) < 2:
        return None, None, "Single cluster"

    sil = silhouette_score(X_clean, labels_clean)
    ch = calinski_harabasz_score(X_clean, labels_clean)
    return sil, ch, "OK"

## Angle-based start/end trail removal (index-tracked)
Same subroutine as before, but now generalised to run on **any** algorithm's clusters (DBSCAN, KMeans, or DePDDP), not just KMeans/DePDDP. Original indices are carried through so we can pull the matching scaled feature rows later for CH scoring.

In [ ]:
def calculate_angle(p1, p2, p3):
    a = np.linalg.norm(p2 - p3)
    b = np.linalg.norm(p1 - p3)
    c = np.linalg.norm(p1 - p2)

    if a == 0 or c == 0:
        return 180

    cos_theta = (a**2 + c**2 - b**2) / (2 * a * c)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))


def remove_start_trail(points, times, idx, threshold=ANGLE_THRESHOLD):
    order = np.argsort(times)
    points, times, idx = points[order], times[order], idx[order]

    while len(points) >= 3:
        angle = calculate_angle(points[0], points[1], points[2])
        if angle > threshold:
            points, times, idx = points[1:], times[1:], idx[1:]
        else:
            break

    return points, times, idx


def remove_end_trail(points, times, idx, threshold=ANGLE_THRESHOLD):
    order = np.argsort(times)[::-1]
    points, times, idx = points[order], times[order], idx[order]

    while len(points) >= 3:
        angle = calculate_angle(points[0], points[1], points[2])
        if angle > threshold:
            points, times, idx = points[1:], times[1:], idx[1:]
        else:
            break

    return points, times, idx


def clean_cluster_trails(points, times, idx, threshold=ANGLE_THRESHOLD):
    points, times, idx = remove_start_trail(points, times, idx, threshold)
    points, times, idx = remove_end_trail(points, times, idx, threshold)
    return points, times, idx


def clean_all_clusters(utm, times, labels, min_points=MIN_CLUSTER_POINTS,
                        threshold=ANGLE_THRESHOLD):
    """
    Runs angle-based start/end trail removal on EVERY non-noise cluster
    produced by one algorithm (DBSCAN / KMeans / DePDDP alike).

    Returns the pooled cleaned points/labels/original-indices (for overall
    CH scoring) plus a per-cluster dict (used later only for the winning
    algorithm's injection + hull step).
    """
    unique_clusters = sorted(set(labels) - {-1})

    cleaned_utm_list, cleaned_labels_list, cleaned_idx_list = [], [], []
    per_cluster = {}

    for cid in unique_clusters:
        mask = labels == cid
        idx_full = np.where(mask)[0]
        pts, tms = utm[mask], times[mask]

        if len(pts) < min_points:
            continue

        c_pts, c_tms, c_idx = clean_cluster_trails(pts, tms, idx_full, threshold)

        if len(c_pts) < min_points:
            continue

        per_cluster[cid] = (c_pts, c_tms, c_idx)
        cleaned_utm_list.append(c_pts)
        cleaned_labels_list.append(np.full(len(c_pts), cid))
        cleaned_idx_list.append(c_idx)

    if not cleaned_utm_list:
        return None, None, None, {}

    return (np.vstack(cleaned_utm_list),
            np.concatenate(cleaned_labels_list),
            np.concatenate(cleaned_idx_list),
            per_cluster)

## Concave hull (adaptive tolerance = estimate() x 3) + polylabel centroid, radius, point injection
The tolerance is **not** a hardcoded absolute number - it's `ch.estimate()` (the library's automatic characteristic-distance estimate for that specific cluster's point density) multiplied by 3. A fixed absolute tol produced badly jagged, spiky hulls once points were injected at various densities; scaling relative to each cluster's own density keeps the boundary smooth regardless of point spacing.

In [ ]:
def find_area_per_utm(utm_points, tol_multiplier=HULL_TOL_MULTIPLIER):
    ch = ConcaveHull()
    ch.loadpoints(utm_points)
    tol = ch.estimate() * tol_multiplier   # adaptive to this cluster's point density
    ch.calculatehull(tol=tol)

    hull_coords = np.array(ch.hull.exterior.coords[:])
    poly = Polygon(hull_coords)

    area = poly.area
    perimeter = poly.length

    try:
        coords = [list(c) for c in poly.exterior.coords]
        pole = polylabel([coords], 0.6)
        cx, cy = (pole.x, pole.y) if hasattr(pole, "x") else (pole[0], pole[1])
        centroid = np.array([cx, cy])
    except Exception:
        cgeom = poly.centroid
        centroid = np.array([cgeom.x, cgeom.y])

    return {
        "Polygon_UTM": hull_coords,
        "Area_m2": area,
        "Perimeter_m": perimeter,
        "Centroid_UTM": centroid
    }


def compute_radius(area_m2, n_points):
    return np.sqrt(area_m2 / (n_points * np.pi))


def inject_points(points, d):
    diag = d / np.sqrt(2)
    injected = []
    for x, y in points:
        injected.extend([
            [x + d, y], [x - d, y],
            [x, y + d], [x, y - d],
            [x + diag, y + diag], [x + diag, y - diag],
            [x - diag, y + diag], [x - diag, y - diag]
        ])
    return np.vstack([points, np.array(injected)])

## Linear interpolation (5-second steps) BEFORE inject point step
Runs on the *cleaned* cluster points (post angle-trail-removal), right before the inject-point step. Points are sorted by time; for every consecutive pair, the time gap is computed and `gap // step` new points are linearly interpolated at fixed 5-second intervals between them. Original points are always retained (never replaced). The resulting augmented set (cleaned points + interpolated points, sorted by time) is what gets passed to `inject_points()`. Note: `r` (used for `d = 0.75r` in the inject-point step) is still computed from the cleaned cluster points only (before interpolation) - interpolation does not affect `r`.

In [ ]:
def linear_interpolate_cluster(points, times, step=INTERP_STEP_SECONDS):
    """
    Densifies a single cluster's points by linear interpolation in time.

    points : (N,2) array of UTM coords for the cleaned cluster
    times  : (N,) array of the matching timestamps (seconds)
    step   : interpolation step in seconds (default INTERP_STEP_SECONDS)

    For every consecutive pair (after sorting by time), the time gap is
    computed and `gap // step` new points are placed at t1+step, t1+2*step, ...
    up to (and possibly including, if the gap divides evenly) t2. Original
    points are always kept - interpolated points are inserted between them,
    never in place of them.

    Returns (augmented_points, augmented_times), both sorted by time.
    """
    order = np.argsort(times)
    pts_sorted = points[order]
    tms_sorted = times[order]

    if len(pts_sorted) < 2:
        return pts_sorted, tms_sorted

    out_points = [pts_sorted[0]]
    out_times = [tms_sorted[0]]

    for i in range(len(pts_sorted) - 1):
        p1, p2 = pts_sorted[i], pts_sorted[i + 1]
        t1, t2 = tms_sorted[i], tms_sorted[i + 1]
        gap = t2 - t1

        if gap > step:
            n_interp = int(gap // step)   # e.g. 30s gap, 5s step -> 6 interpolated points
            for k in range(1, n_interp + 1):
                frac = (k * step) / gap
                frac = min(frac, 1.0)
                interp_point = p1 + frac * (p2 - p1)
                interp_time = t1 + k * step
                out_points.append(interp_point)
                out_times.append(interp_time)

        out_points.append(p2)
        out_times.append(t2)

    out_points = np.array(out_points)
    out_times = np.array(out_times)

    final_order = np.argsort(out_times)
    return out_points[final_order], out_times[final_order]

## Plot helper

In [ ]:
def save_cluster_plot(labels, utm, path, title):
    plt.figure(figsize=(7, 7))
    for label in set(labels):
        mask = labels == label
        if label == -1:
            plt.scatter(utm[mask, 0], utm[mask, 1], c='black', s=10, label='Noise')
        else:
            plt.scatter(utm[mask, 0], utm[mask, 1], s=10, label=f'Cluster {label}')
    plt.title(title)
    plt.legend()
    plt.gca().set_aspect("equal")
    plt.savefig(path)
    plt.close()

## Main pipeline

**Logic:**
1. Run DBSCAN at auto (knee) eps -> get `n_clusters_dbscan`.
2. Angle-clean DBSCAN's clusters regardless.
3. If `n_clusters_dbscan == 1` -> DBSCAN is automatically the winner (KMeans/DePDDP don't run).
4. Else -> run KMeans and DePDDP with `k = n_clusters_dbscan`, angle-clean their clusters too, compute overall CH for all three, pick the highest as winner.
5. **Minimum cluster size filter:** for the winning algorithm only, each of its individual clusters is checked - if it has fewer than `MIN_HULL_CLUSTER_POINTS` (15) points, it is eliminated and logged, and does not proceed to injection/hull.
6. Point injection + concave hull (adaptive tol) + centroid run only on the winning algorithm's clusters that survive the size filter.
7. Two Excel files are saved: the final per-cluster result, and a workbook with a CH comparison sheet (across all three algorithms) plus an Eliminated_Clusters sheet (clusters dropped for being too small).

In [ ]:
main_results = []          # Excel 1: final per-cluster result (winning algo only)
comparison_results = []    # Excel 2 (sheet 1): per-field CH scores of all 3 algos
eliminated_results = []    # Excel 2 (sheet 2): clusters of the winning algo dropped for being too small

files = sorted([f for f in os.listdir(DATA_FOLDER) if f.endswith(".csv")])[:MAX_FILES]

for file in files:

    print("\n==============================")
    print("FILE:", file)

    # ---------- load & clean ----------
    df = pd.read_csv(os.path.join(DATA_FOLDER, file))
    df.columns = [c.strip() for c in df.columns]
    df.rename(columns={"Latitude": "Lat", "Longitude": "Lon", "TimeStamp": "Time"}, inplace=True)

    if not {"Lat", "Lon"}.issubset(df.columns):
        continue

    cols = ["Lat", "Lon"]
    if "Time" in df.columns:
        cols.insert(0, "Time")

    df = df[cols].apply(pd.to_numeric, errors="coerce").dropna()
    if len(df) < 20:
        continue

    utm, X, times = prepare_data(df)

    plot_id = os.path.splitext(file)[0]
    field_folder = os.path.join(OUTPUT_FOLDER, plot_id)
    os.makedirs(field_folder, exist_ok=True)

    # ---------- STEP 1: DBSCAN at auto (knee) eps ----------
    nbrs = NearestNeighbors(n_neighbors=NEIGHBORS).fit(X)
    distances, _ = nbrs.kneighbors(X)
    kth_distances = np.sort(distances[:, NEIGHBORS - 1])

    kneedle = KneeLocator(range(len(kth_distances)), kth_distances,
                           S=1.0, curve="convex", direction="increasing")
    optimal_eps = kneedle.knee_y

    if optimal_eps is None:
        print("No knee found, skipping file.")
        continue

    labels_dbscan = DBSCAN(eps=float(optimal_eps), min_samples=NEIGHBORS).fit_predict(X)
    n_clusters_dbscan = len(set(labels_dbscan) - {-1})

    print(f"DBSCAN eps={optimal_eps:.4f} -> clusters={n_clusters_dbscan}")

    save_cluster_plot(labels_dbscan, utm,
                       os.path.join(field_folder, "dbscan_clusters.png"),
                       f"{plot_id} | DBSCAN | eps={optimal_eps:.4f}")

    if n_clusters_dbscan < 1:
        print("DBSCAN found no clusters, skipping file.")
        continue

    # ---------- angle removal on DBSCAN clusters (always happens) ----------
    db_utm_all, db_labels_all, db_idx_all, db_per_cluster = clean_all_clusters(utm, times, labels_dbscan)

    if db_utm_all is None:
        print("All DBSCAN clusters too small after cleaning, skipping file.")
        continue

    _, ch_db, _ = compute_scores(X[db_idx_all], db_labels_all)

    ch_scores = {"DBSCAN": ch_db, "KMeans": None, "DePDDP": None}
    per_cluster_by_algo = {"DBSCAN": db_per_cluster, "KMeans": None, "DePDDP": None}

    # ---------- STEP 2: branch on number of DBSCAN clusters ----------
    if n_clusters_dbscan == 1:
        # Flowchart "Yes" path -> ONLY DBSCAN is relevant, it wins by
        # default (no KMeans / DePDDP run, no CH competition needed).
        best_algo = "DBSCAN"

    else:
        # Flowchart "No" path -> also run KMeans & DePDDP with the same k,
        # angle-clean ALL THREE algorithms' clusters, then compare CH.
        try:
            labels_km = KMeans(n_clusters=n_clusters_dbscan, random_state=42, n_init=10).fit_predict(X)
            km_utm_all, km_labels_all, km_idx_all, km_per_cluster = clean_all_clusters(utm, times, labels_km)
            if km_utm_all is not None:
                _, ch_km, _ = compute_scores(X[km_idx_all], km_labels_all)
                ch_scores["KMeans"] = ch_km
                per_cluster_by_algo["KMeans"] = km_per_cluster
        except Exception as e:
            print("KMeans failed:", e)

        try:
            model = DePDDP(decomposition_method="pca", max_clusters_number=n_clusters_dbscan,
                            bandwidth_scale=0.2, percentile=0.05, min_sample_split=15)
            labels_dep = model.fit_predict(X)
            dep_utm_all, dep_labels_all, dep_idx_all, dep_per_cluster = clean_all_clusters(utm, times, labels_dep)
            if dep_utm_all is not None:
                _, ch_dep, _ = compute_scores(X[dep_idx_all], dep_labels_all)
                ch_scores["DePDDP"] = ch_dep
                per_cluster_by_algo["DePDDP"] = dep_per_cluster
        except Exception as e:
            print("DePDDP failed:", e)

        valid = {k: v for k, v in ch_scores.items() if v is not None}
        if not valid:
            print("No algorithm produced a valid CH score, skipping file.")
            continue

        best_algo = max(valid, key=valid.get)

    print(f"CH scores -> {ch_scores}")
    print(f"BEST ALGORITHM: {best_algo}")

    comparison_results.append({
        "File": file,
        "N_Clusters_DBSCAN": n_clusters_dbscan,
        "CH_DBSCAN": ch_scores["DBSCAN"],
        "CH_KMeans": ch_scores["KMeans"],
        "CH_DePDDP": ch_scores["DePDDP"],
        "Best_Algorithm": best_algo
    })

    # ---------- STEP 3: injection + hull ONLY for the winning algorithm ----------
    best_per_cluster = per_cluster_by_algo[best_algo]

    for cid, (pts, tms, idx) in best_per_cluster.items():

        # ---- minimum cluster size filter (post best-algorithm selection) ----
        #if len(pts) < MIN_HULL_CLUSTER_POINTS:
            #print(f"Cluster {cid} eliminated: only {len(pts)} points (< {MIN_HULL_CLUSTER_POINTS}). Skipping hull.")
            #eliminated_results.append({
              #  "File": file,
                #"Cluster_ID": cid,
                #"Best_Algorithm": best_algo,
               # "Point_Count": len(pts),
               # "Reason": f"Fewer than {MIN_HULL_CLUSTER_POINTS} points after cleaning"
            #})
            #continue

        try:
            cleaned_hull = find_area_per_utm(pts)

            n_points = len(pts)
            r = 0.75*compute_radius(cleaned_hull["Area_m2"], n_points)

            # ---- NEW: linear interpolation (5-sec steps) BEFORE inject point step ----
            # r above is computed from the cleaned cluster points (pts) only, as before.
            # The augmented (cleaned + interpolated) points are used ONLY for injection/hull.
            interp_pts, interp_tms = linear_interpolate_cluster(pts, tms, step=INTERP_STEP_SECONDS)
            n_interp_points = len(interp_pts)

            augmented_points = inject_points(interp_pts, r)
            injected_hull = find_area_per_utm(augmented_points)

            # -- plot --
            plt.figure(figsize=(7, 7))
            plt.scatter(pts[:, 0], pts[:, 1], c='blue', s=8, label='Cleaned Points')
            plt.scatter(interp_pts[:, 0], interp_pts[:, 1], c='purple', s=4, alpha=0.6, label='Interpolated Points')
            plt.scatter(augmented_points[len(interp_pts):, 0], augmented_points[len(interp_pts):, 1],
                        c='orange', s=2, alpha=0.5, label='Injected Points')
            plt.plot(injected_hull["Polygon_UTM"][:, 0], injected_hull["Polygon_UTM"][:, 1],
                     c='red', linewidth=2, label='Final Hull')
            plt.scatter(*injected_hull["Centroid_UTM"], marker='x', c='green', s=80, label='Centroid')
            plt.title(f"{plot_id} | {best_algo} | Cluster {cid}")
            plt.legend()
            plt.gca().set_aspect("equal")
            plt.savefig(os.path.join(field_folder, f"{best_algo}_cluster_{cid}_final.png"))
            plt.close()

            main_results.append({
                "File": file,
                "Cluster_ID": cid,
                "Best_Algorithm": best_algo,
                "CH_Score": ch_scores[best_algo],
                "Cleaned_Points": n_points,
                "Interpolated_Points": n_interp_points,
                "Radius_r": r,
                "Cleaned_Area_m2": cleaned_hull["Area_m2"],
                "Cleaned_Perimeter_m": cleaned_hull["Perimeter_m"],
                "Area_m2": injected_hull["Area_m2"],
                "Perimeter_m": injected_hull["Perimeter_m"],
                "Centroid_X": injected_hull["Centroid_UTM"][0],
                "Centroid_Y": injected_hull["Centroid_UTM"][1],
            })

        except Exception as e:
            print(f"Hull/injection failed for cluster {cid}:", e)


FILE: H1260624_0813-0818.csv
DBSCAN eps=0.7874 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': None}
BEST ALGORITHM: DBSCAN

FILE: H1260624_0819-0823.csv
DBSCAN eps=0.5586 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': None}
BEST ALGORITHM: DBSCAN

FILE: H1260624_0824-0828.csv
DBSCAN eps=0.6265 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': None}
BEST ALGORITHM: DBSCAN

FILE: H1260624_0829-0836.csv
DBSCAN eps=0.6548 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': None}
BEST ALGORITHM: DBSCAN

FILE: H1260624_0836-0842.csv
DBSCAN eps=0.7667 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': None}
BEST ALGORITHM: DBSCAN

FILE: H1260624_0843-0849.csv
DBSCAN eps=0.4619 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': None}
BEST ALGORITHM: DBSCAN

FILE: H1260624_0850-0856.csv
DBSCAN eps=0.6705 -> clusters=1
CH scores -> {'DBSCAN': None, 'KMeans': None, 'DePDDP': 

## Save both Excel outputs

In [ ]:
df_main = pd.DataFrame(main_results)
df_main.to_excel(os.path.join(OUTPUT_FOLDER, "final_field_results.xlsx"), index=False)

df_compare = pd.DataFrame(comparison_results)
df_eliminated = pd.DataFrame(eliminated_results)
with pd.ExcelWriter(os.path.join(OUTPUT_FOLDER, "algorithm_ch_comparison.xlsx")) as writer:
    df_compare.to_excel(writer, sheet_name="CH_Comparison", index=False)
    df_eliminated.to_excel(writer, sheet_name="Eliminated_Clusters", index=False)

print("\n✅ Saved:", os.path.join(OUTPUT_FOLDER, "final_field_results.xlsx"))
print("✅ Saved:", os.path.join(OUTPUT_FOLDER, "algorithm_ch_comparison.xlsx"), "(CH_Comparison + Eliminated_Clusters sheets)")


✅ Saved: /content/drive/MyDrive/MergedPipeline(0.75r)newdata/24thJune/tol(2.5)/final_field_results.xlsx
✅ Saved: /content/drive/MyDrive/MergedPipeline(0.75r)newdata/24thJune/tol(2.5)/algorithm_ch_comparison.xlsx (CH_Comparison + Eliminated_Clusters sheets)
